# Complete Case Analysis (CCA)

**Complete Case Analysis** is the simplest possible way to "handle" missing data: just **drop every row** that has a missing value in any of the columns being used. It's only safe when missing values are rare *and* basically random — this notebook checks both conditions before deciding whether CCA is actually a reasonable choice here.

## 1. Setup and Data

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
df = pd.read_csv('data_science_job.csv')

FileNotFoundError: [Errno 2] No such file or directory: 'data_science_job.csv'

In [ ]:
df.head()

## 2. How Much Data Is Actually Missing?

`.isnull().mean() * 100` gives the percentage missing per column — the key number for deciding if CCA is even viable. Dropping rows for a column missing 1% of its values barely costs anything; dropping rows for one missing 60% would gut the dataset.

In [ ]:
df.isnull().mean()*100

In [ ]:
df.shape

### Picking Only the "Safe to Drop" Columns

CCA is only applied to columns missing **less than 5%** of their values (and at least *some* missing, otherwise there's nothing to do) — a common rule-of-thumb threshold below which dropping rows costs relatively little data.

In [ ]:
cols = [var for var in df.columns if df[var].isnull().mean() < 0.05 and df[var].isnull().mean() > 0]
cols

In [ ]:
df[cols].sample(5)

In [ ]:
df['education_level'].value_counts()

### How Much Data Would Actually Be Lost?

Before committing to CCA, it's worth checking the real cost: what fraction of *all* rows survive once every row with any missing value (across just these low-missingness columns) is dropped?

In [ ]:
len(df[cols].dropna()) / len(df)

In [ ]:
new_df = df[cols].dropna()
df.shape, new_df.shape

## 3. Checking Whether Dropped Rows Look Different — Numeric Columns

The real risk with CCA: if rows with missing data are systematically *different* from the rest (not missing at random), dropping them biases the remaining dataset. Overlaying each numeric column's distribution **before** (red) and **after** (green) CCA reveals whether that happened — if the two curves look almost identical, CCA didn't introduce meaningful bias for that column.

In [ ]:
new_df.hist(bins=50, density=True, figsize=(12, 12))
plt.show()

In [ ]:
fig = plt.figure()
ax = fig.add_subplot(111)

# original data
df['training_hours'].hist(bins=50, ax=ax, density=True, color='red')

# data after cca, the argument alpha makes the color transparent, so we can
# see the overlay of the 2 distributions
new_df['training_hours'].hist(bins=50, ax=ax, color='green', density=True, alpha=0.8)

In [ ]:
fig = plt.figure()
ax = fig.add_subplot(111)

# original data
df['training_hours'].plot.density(color='red')

# data after cca
new_df['training_hours'].plot.density(color='green')

In [ ]:
fig = plt.figure()
ax = fig.add_subplot(111)

# original data
df['city_development_index'].hist(bins=50, ax=ax, density=True, color='red')

# data after cca, the argument alpha makes the color transparent, so we can
# see the overlay of the 2 distributions
new_df['city_development_index'].hist(bins=50, ax=ax, color='green', density=True, alpha=0.8)

In [ ]:
fig = plt.figure()
ax = fig.add_subplot(111)

# original data
df['city_development_index'].plot.density(color='red')

# data after cca
new_df['city_development_index'].plot.density(color='green')

In [ ]:
fig = plt.figure()
ax = fig.add_subplot(111)

# original data
df['experience'].hist(bins=50, ax=ax, density=True, color='red')

# data after cca, the argument alpha makes the color transparent, so we can
# see the overlay of the 2 distributions
new_df['experience'].hist(bins=50, ax=ax, color='green', density=True, alpha=0.8)

In [ ]:
fig = plt.figure()
ax = fig.add_subplot(111)

# original data
df['experience'].plot.density(color='red')

# data after cca
new_df['experience'].plot.density(color='green')

## 4. Checking Whether Dropped Rows Look Different — Categorical Columns

Same idea, but for a categorical column there's no distribution curve to overlay — instead, compare the **proportion of each category** before vs. after CCA. A category whose share shifts noticeably after dropping rows is a sign that missingness wasn't random with respect to that category.

In [ ]:
temp = pd.concat([
            # percentage of observations per category, original data
            df['enrolled_university'].value_counts() / len(df),

            # percentage of observations per category, cca data
            new_df['enrolled_university'].value_counts() / len(new_df)
        ],
        axis=1)

# add column names
temp.columns = ['original', 'cca']

temp

In [ ]:
temp = pd.concat([
            # percentage of observations per category, original data
            df['education_level'].value_counts() / len(df),

            # percentage of observations per category, cca data
            new_df['education_level'].value_counts() / len(new_df)
        ],
        axis=1)

# add column names
temp.columns = ['original', 'cca']

temp

### Visualizing the Categorical Comparison

The tables above are easy to skim for two categories, but a bar chart makes any shift much more obvious at a glance, especially with more categories.

In [ ]:
temp.plot(kind='bar', figsize=(8, 4))
plt.title('education_level: Original vs. After CCA')
plt.ylabel('Proportion of rows')
plt.show()


## Summary

| Task | Method |
|---|---|
| Check missingness per column | `df.isnull().mean() * 100` |
| Pick low-missingness columns | list comprehension filtering on `.isnull().mean()` |
| Drop incomplete rows | `df[cols].dropna()` |
| Check numeric distributions stayed the same | overlay `.hist()` / `.plot.density()` before vs. after |
| Check categorical proportions stayed the same | compare `.value_counts() / len(df)` before vs. after |

### When CCA Is (and Isn't) a Reasonable Choice
- **OK to use** — missingness is low (rule of thumb: under ~5%) *and* the before/after comparisons above show no meaningful shift in distribution.
- **Avoid** — missingness is substantial, or the comparisons reveal that rows with missing data look systematically different (e.g. lower-income respondents skip an income question) — dropping them would bias whatever's left.